In [29]:
#!pip install opencv-python
import os
os.environ["QT_QPA_PLATFORM"] = "xcb"
#Gestion de l'affichage pour opencv wailand

In [30]:
import cv2 
import numpy as np
import matplotlib.pyplot as plt

In [31]:
racine_entrainement = "/home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP2/CCPD_Dataset_preproc/train/"
save_dir = "/home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP2/Tranning_Models_TP2"

In [ ]:
def densite_zones(img, line_segmentation, column_segmentation):
    """
    Calcule le vecteur de densité de pixels pour une image binaire
    découpée en (line_segmentation x column_segmentation) zones.
    """

    caracteristic_vector = []

    for bande in np.array_split(img, line_segmentation, axis=0):
        for zone in np.array_split(bande, column_segmentation, axis=1):
#            print(bande.shape, zone.shape)
            densite = np.count_nonzero(zone) / zone.size
            caracteristic_vector.append(densite)

    caracteristic_vector = np.array(caracteristic_vector)

    return caracteristic_vector if caracteristic_vector.size > 0 else np.nan

In [33]:
line_segmentation = 5
column_segmentation = 5

X_train = []
y_train = []

# repertoire pour sauvegarder les profils
os.makedirs(save_dir, exist_ok=True)

for chemin_dossier, sous_dossiers, fichiers in os.walk(racine_entrainement):
    classe = os.path.basename(chemin_dossier)
    resultats = []

    for fichier in fichiers:
        if fichier.lower().endswith((".png")):
            chemin_complet = os.path.join(chemin_dossier, fichier)
            TAILLE_FIXE = (50, 50)

            img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)

            img = cv2.resize(img, TAILLE_FIXE, interpolation=cv2.INTER_NEAREST)


            if img is None:
                continue
            
            profil = densite_zones(img,line_segmentation,column_segmentation)

            X_train.append(profil)
            y_train.append(classe)

# Conversion en tableau numpy
X_train = np.array(X_train)
y_train = np.array(y_train)

# Sauvegarde globale de la base
np.save(os.path.join(save_dir, "X_train.npy"), X_train)
np.save(os.path.join(save_dir, "y_train.npy"), y_train)

print(f"Base d'apprentissage créée : {X_train.shape[0]} images, {X_train.shape[1]} caractéristiques.")


(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (

In [34]:
def distance_(x_test, x_train):
    return np.linalg.norm(x_test - x_train)


from collections import Counter

def predict_KNN(img, X_train, y_train, line_segmentation, column_segmentation, K=5):
    """
    Prédit la classe et les probabilités d'une image donnée avec la méthode KNN.
    """
    # 1️⃣ Calcul du vecteur zoning pour l'image test
    x_test = densite_zones(img, line_segmentation, column_segmentation)

    # 2️⃣ Calcul des distances entre x_test et chaque vecteur de la base
    distances = np.linalg.norm(X_train - x_test, axis=1)

    # 3️⃣ Indices des K plus proches voisins
    indices_voisins = np.argsort(distances)[:K]

    # 4️⃣ Récupération des classes correspondantes
    voisins_classes = y_train[indices_voisins]

    # 5️⃣ Comptage du nombre de voisins par classe
    compte = Counter(voisins_classes)

    # 6️⃣ Calcul du vecteur de probabilités p2(Ci|x) = ki / K
    classes_uniques = np.unique(y_train)
    p2 = np.zeros(len(classes_uniques))
    for i, c in enumerate(classes_uniques):
        p2[i] = compte.get(c, 0) / K

    # 7️⃣ Classe prédite = celle avec la plus grande probabilité
    prediction = classes_uniques[np.argmax(p2)]

    return prediction, p2



def load_base(save_dir):
    X_train = np.load(os.path.join(save_dir, "X_train.npy"))
    y_train = np.load(os.path.join(save_dir, "y_train.npy"))
    return X_train, y_train
        

In [35]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Chargement de la base
X_train, y_train = load_base(save_dir)
line_segmentation = 5
column_segmentation = 5

# Dossiers
racine_test = "/home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP2/CCPD_Dataset_preproc/test/"
save_dir_proba = "/home/julien/Documents/Master_VASA_UTBM/VA52_TP/TP2/Training_Test"

os.makedirs(save_dir_proba, exist_ok=True)

# Pour l’évaluation
y_true, y_pred = [], []

# Parcours des images de test
for chemin_dossier, _, fichiers in os.walk(racine_test):
    classe_reelle = os.path.basename(chemin_dossier)
    if not classe_reelle or classe_reelle == os.path.basename(racine_test):
        continue

    for fichier in fichiers:
        if not fichier.lower().endswith(".png"):
            continue

        chemin_complet = os.path.join(chemin_dossier, fichier)

        TAILLE_FIXE = (50, 50)
        img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, TAILLE_FIXE, interpolation=cv2.INTER_NEAREST)

        if img is None:
            continue

        # --- Prédiction ---
        prediction, probabilites = predict_KNN(img, X_train, y_train,
                                                line_segmentation, column_segmentation, K=5)

        # --- Stockage pour évaluation ---
        y_true.append(classe_reelle)
        y_pred.append(prediction)

        # --- Sauvegarde du vecteur de probas ---
        vec_probabilites = probabilites
        nom_sans_ext = os.path.splitext(fichier)[0]
        name_file = os.path.join(save_dir_proba, nom_sans_ext + "_proba.npy")
        np.save(name_file, vec_probabilites)

# --- Évaluation globale ---
labels = np.unique(y_train)
acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=sorted(labels))

print("\n=== Résultats d'évaluation ===")
print(f"Nombre total d'images testées : {len(y_true)}")
print(f"Taux global de reconnaissance : {acc:.2%}")

# --- Rapport par classe ---
print("\n=== Rapport par classe ===")
report = classification_report(y_true, y_pred, digits=3)
print(report)


(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (10, 10)
(10, 50) (